In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# Data load
print("\nData load ho raha hai...")
parsed_path = "/content/drive/MyDrive/ASAD_Thesis/processed_data/spark_logs_parsed.csv"
df = pd.read_csv(parsed_path)
print(f"Shape: {df.shape}")

# Features
le_app = LabelEncoder()
le_container = LabelEncoder()
df['app_id_enc'] = le_app.fit_transform(df['app_id'])
df['container_id_enc'] = le_container.fit_transform(df['container_id'])
df['binary_label'] = df['label'].apply(lambda x: 0 if x == 0 else 1)

features = ['cluster_id', 'app_id_enc', 'container_id_enc']
X = df[features].values
y = df['binary_label'].values

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Sirf normal training data
X_train_normal = X_train[y_train == 0]

print(f"\nX_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"Normal train samples: {len(X_train_normal):,}")
print("\nSab ready hai! ✅")

Mounted at /content/drive
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Data load ho raha hai...
Shape: (1661830, 6)

X_train: (1329464, 3)
X_test: (332366, 3)
Normal train samples: 1,327,644

Sab ready hai! ✅


In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Layer
from tensorflow.keras.callbacks import EarlyStopping

print("VAE banana shuru...")

input_dim = 3
latent_dim = 2

# ============================================
# Sampling Layer — proper Keras layer
# ============================================
class Sampling(Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# ============================================
# ENCODER
# ============================================
inputs = Input(shape=(input_dim,))
h = Dense(16, activation='relu')(inputs)
z_mean = Dense(latent_dim)(h)
z_log_var = Dense(latent_dim)(h)
z = Sampling()([z_mean, z_log_var])

encoder = Model(inputs, [z_mean, z_log_var, z], name='encoder')

# ============================================
# DECODER
# ============================================
latent_inputs = Input(shape=(latent_dim,))
h_decoded = Dense(16, activation='relu')(latent_inputs)
outputs = Dense(input_dim, activation='linear')(h_decoded)

decoder = Model(latent_inputs, outputs, name='decoder')

# ============================================
# VAE — Custom Model with loss inside
# ============================================
class VAE(Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def call(self, x):
        z_mean, z_log_var, z = self.encoder(x)
        reconstructed = self.decoder(z)
        # KL Loss
        kl_loss = -0.5 * tf.reduce_mean(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        self.add_loss(kl_loss)
        return reconstructed

vae = VAE(encoder, decoder)
vae.compile(optimizer='adam', loss='mse')

# Summary
print(f"Encoder params: {encoder.count_params()}")
print(f"Decoder params: {decoder.count_params()}")

# Train
print("\nVAE Training shuru...")
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True)

history = vae.fit(
    X_train_normal, X_train_normal,
    epochs=20,
    batch_size=512,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print("\nVAE Training complete! ✅")

VAE banana shuru...
Encoder params: 132
Decoder params: 99

VAE Training shuru...
Epoch 1/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - loss: 0.6653 - val_loss: 0.6162
Epoch 2/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.6083 - val_loss: 0.6005
Epoch 3/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.6015 - val_loss: 0.5961
Epoch 4/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.6007 - val_loss: 0.5948
Epoch 5/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.5994 - val_loss: 0.5988
Epoch 6/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.5993 - val_loss: 0.5974
Epoch 7/20
2334/2334 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.5991 - val_loss: 0.5963

VAE Training complete! ✅


In [4]:
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print("VAE Evaluation...")

# Reconstruction error calculate karo
X_test_reconstructed = vae(X_test, training=False).numpy()
mse_vae = np.mean(np.power(X_test - X_test_reconstructed, 2), axis=1)

print(f"Normal MSE mean:  {mse_vae[y_test==0].mean():.6f}")
print(f"Anomaly MSE mean: {mse_vae[y_test==1].mean():.6f}")

# Threshold — 95th percentile of normal
threshold_vae = np.percentile(mse_vae[y_test==0], 95)
print(f"Threshold (95th percentile): {threshold_vae:.6f}")

# Predict
y_pred_vae = (mse_vae > threshold_vae).astype(int)

print("\n=== VAE Results ===")
print(classification_report(y_test, y_pred_vae,
                            target_names=['Normal', 'Anomaly']))

roc_auc_vae = roc_auc_score(y_test, mse_vae)
print(f"ROC-AUC Score: {roc_auc_vae:.4f}")

cm = confusion_matrix(y_test, y_pred_vae)
print(f"\nConfusion Matrix:")
print(cm)

VAE Evaluation...
Normal MSE mean:  0.231853
Anomaly MSE mean: 0.610851
Threshold (95th percentile): 1.110775

=== VAE Results ===
              precision    recall  f1-score   support

      Normal       1.00      0.95      0.97    331911
     Anomaly       0.00      0.16      0.01       455

    accuracy                           0.95    332366
   macro avg       0.50      0.56      0.49    332366
weighted avg       1.00      0.95      0.97    332366

ROC-AUC Score: 0.7700

Confusion Matrix:
[[315315  16596]
 [   380     75]]


In [5]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Dropout
from tensorflow.keras.optimizers import Adam
import numpy as np

print("GAN banana shuru...")

input_dim = 3
latent_dim = 16

# ============================================
# GENERATOR
# ============================================
def build_generator():
    model = Sequential([
        Dense(16, input_dim=latent_dim),
        LeakyReLU(0.2),
        Dense(32),
        LeakyReLU(0.2),
        Dense(input_dim, activation='linear')
    ], name='generator')
    return model

# ============================================
# DISCRIMINATOR
# ============================================
def build_discriminator():
    model = Sequential([
        Dense(32, input_dim=input_dim),
        LeakyReLU(0.2),
        Dropout(0.3),
        Dense(16),
        LeakyReLU(0.2),
        Dense(1, activation='sigmoid')
    ], name='discriminator')
    return model

generator = build_generator()
discriminator = build_discriminator()

# Discriminator compile
discriminator.compile(optimizer=Adam(0.0002),
                      loss='binary_crossentropy',
                      metrics=['accuracy'])

# GAN — discriminator frozen during generator training
discriminator.trainable = False
gan_input = Input(shape=(latent_dim,))
gan_output = discriminator(generator(gan_input))
gan = Model(gan_input, gan_output, name='gan')
gan.compile(optimizer=Adam(0.0002), loss='binary_crossentropy')

print("Generator params:", generator.count_params())
print("Discriminator params:", discriminator.count_params())

# ============================================
# GAN Training Loop
# ============================================
EPOCHS = 50
BATCH_SIZE = 512

print(f"\nGAN Training shuru ({EPOCHS} epochs)...")

for epoch in range(EPOCHS):
    # Random normal samples select karo
    idx = np.random.randint(0, X_train_normal.shape[0], BATCH_SIZE)
    real_samples = X_train_normal[idx]

    # Fake samples generate karo
    noise = np.random.normal(0, 1, (BATCH_SIZE, latent_dim))
    fake_samples = generator.predict(noise, verbose=0)

    # Labels
    real_labels = np.ones((BATCH_SIZE, 1))
    fake_labels = np.zeros((BATCH_SIZE, 1))

    # Discriminator train
    d_loss_real = discriminator.train_on_batch(real_samples, real_labels)
    d_loss_fake = discriminator.train_on_batch(fake_samples, fake_labels)

    # Generator train
    noise = np.random.normal(0, 1, (BATCH_SIZE, latent_dim))
    g_loss = gan.train_on_batch(noise, real_labels)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"D_real: {d_loss_real[0]:.4f} | "
              f"D_fake: {d_loss_fake[0]:.4f} | "
              f"G_loss: {g_loss:.4f}")

print("\nGAN Training complete! ✅")

GAN banana shuru...
Generator params: 915
Discriminator params: 673

GAN Training shuru (50 epochs)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Epoch 10/50 | D_real: 0.6370 | D_fake: 0.6468 | G_loss: 0.5872
Epoch 20/50 | D_real: 0.6469 | D_fake: 0.6519 | G_loss: 0.5801
Epoch 30/50 | D_real: 0.6524 | D_fake: 0.6561 | G_loss: 0.5733
Epoch 40/50 | D_real: 0.6584 | D_fake: 0.6614 | G_loss: 0.5673
Epoch 50/50 | D_real: 0.6644 | D_fake: 0.6672 | G_loss: 0.5607

GAN Training complete! ✅


In [6]:
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print("GAN Evaluation...")

# Discriminator anomaly detector ki tarah use karo
# Normal data = high score, Anomaly = low score
d_scores = discriminator.predict(X_test, batch_size=512, verbose=0).flatten()

# Anomaly score = 1 - discriminator score
anomaly_scores = 1 - d_scores

print(f"Normal anomaly score mean:  {anomaly_scores[y_test==0].mean():.4f}")
print(f"Anomaly anomaly score mean: {anomaly_scores[y_test==1].mean():.4f}")

# Threshold — 95th percentile
threshold_gan = np.percentile(anomaly_scores[y_test==0], 95)
print(f"Threshold: {threshold_gan:.4f}")

# Predict
y_pred_gan = (anomaly_scores > threshold_gan).astype(int)

print("\n=== GAN Results ===")
print(classification_report(y_test, y_pred_gan,
                            target_names=['Normal', 'Anomaly']))

roc_auc_gan = roc_auc_score(y_test, anomaly_scores)
print(f"ROC-AUC Score: {roc_auc_gan:.4f}")

cm = confusion_matrix(y_test, y_pred_gan)
print(f"\nConfusion Matrix:")
print(cm)

# ============================================
# FINAL COMPLETE COMPARISON
# ============================================
print("\n" + "=" * 60)
print("COMPLETE RESULTS — ALL 5 MODELS")
print("=" * 60)
print(f"\n{'Model':<22} {'ROC-AUC':>10} {'Recall':>10} {'F1':>10}")
print("-" * 55)

all_results = [
    ("CNN Classifier",    0.9978, 1.00, 0.16),
    ("Random Forest",     0.9902, 0.89, 0.44),
    ("LSTM Autoencoder",  0.9822, 0.93, 0.05),
    ("Isolation Forest",  0.9559, 0.04, 0.04),
    ("VAE",               0.7700, 0.16, 0.01),
    ("GAN",               roc_auc_gan, None, None),
    ("One-Class SVM",     0.2388, 0.01, 0.00),
]

for model, roc, recall, f1 in all_results:
    recall_str = f"{recall:.2f}" if recall is not None else "TBD"
    f1_str = f"{f1:.2f}" if f1 is not None else "TBD"
    print(f"{model:<22} {roc:>10.4f} {recall_str:>10} {f1_str:>10}")

GAN Evaluation...
Normal anomaly score mean:  0.3607
Anomaly anomaly score mean: 0.2687
Threshold: 0.4319

=== GAN Results ===
              precision    recall  f1-score   support

      Normal       1.00      0.95      0.97    331911
     Anomaly       0.00      0.12      0.01       455

    accuracy                           0.95    332366
   macro avg       0.50      0.54      0.49    332366
weighted avg       1.00      0.95      0.97    332366

ROC-AUC Score: 0.2479

Confusion Matrix:
[[315405  16506]
 [   399     56]]

COMPLETE RESULTS — ALL 5 MODELS

Model                     ROC-AUC     Recall         F1
-------------------------------------------------------
CNN Classifier             0.9978       1.00       0.16
Random Forest              0.9902       0.89       0.44
LSTM Autoencoder           0.9822       0.93       0.05
Isolation Forest           0.9559       0.04       0.04
VAE                        0.7700       0.16       0.01
GAN                        0.2479        TBD

In [7]:
import json, os
import numpy as np

save_dir = "/content/drive/MyDrive/ASAD_Thesis/results"
os.makedirs(save_dir, exist_ok=True)

# Complete results
all_results = {
    "Supervised": {
        "CNN_Classifier":   {"ROC_AUC": 0.9978, "Recall": 1.00, "F1": 0.16},
        "Random_Forest":    {"ROC_AUC": 0.9902, "Recall": 0.89, "F1": 0.44},
    },
    "Deep_Learning_Unsupervised": {
        "LSTM_Autoencoder": {"ROC_AUC": 0.9822, "Recall": 0.93, "F1": 0.05},
    },
    "Traditional_Unsupervised": {
        "Isolation_Forest": {"ROC_AUC": 0.9559, "Recall": 0.04, "F1": 0.04},
        "OneClass_SVM":     {"ROC_AUC": 0.2388, "Recall": 0.01, "F1": 0.00},
    },
    "Generative": {
        "VAE":              {"ROC_AUC": 0.7700, "Recall": 0.16, "F1": 0.01},
        "GAN":              {"ROC_AUC": 0.2479, "Recall": 0.12, "F1": 0.01},
    }
}

with open(f"{save_dir}/complete_results.json", "w") as f:
    json.dump(all_results, f, indent=4)

print("Results saved! ✅")
print(json.dumps(all_results, indent=4))

Results saved! ✅
{
    "Supervised": {
        "CNN_Classifier": {
            "ROC_AUC": 0.9978,
            "Recall": 1.0,
            "F1": 0.16
        },
        "Random_Forest": {
            "ROC_AUC": 0.9902,
            "Recall": 0.89,
            "F1": 0.44
        }
    },
    "Deep_Learning_Unsupervised": {
        "LSTM_Autoencoder": {
            "ROC_AUC": 0.9822,
            "Recall": 0.93,
            "F1": 0.05
        }
    },
    "Traditional_Unsupervised": {
        "Isolation_Forest": {
            "ROC_AUC": 0.9559,
            "Recall": 0.04,
            "F1": 0.04
        },
        "OneClass_SVM": {
            "ROC_AUC": 0.2388,
            "Recall": 0.01,
            "F1": 0.0
        }
    },
    "Generative": {
        "VAE": {
            "ROC_AUC": 0.77,
            "Recall": 0.16,
            "F1": 0.01
        },
        "GAN": {
            "ROC_AUC": 0.2479,
            "Recall": 0.12,
            "F1": 0.01
        }
    }
}
